# SIN393 – Classificação de doenças em folhas de arroz com Deep Learning
**Disciplina:** SIN 393 – Introdução à Visão Computacional (UFV-CRP, 2026) — Prof. João Fernando Mari

**Dataset:** [Rice Disease Dataset (Kaggle)](https://www.kaggle.com/datasets/anshulm257/rice-disease-dataset)

**Questão experimental:** comparar uma CNN (ResNet-50) com um Transformer (ViT-B/16), ambos com *transfer learning* do ImageNet, e medir o impacto do *data augmentation* em cada um (4 experimentos).

> No Colab: **Ambiente de execução → Alterar o tipo de ambiente → GPU (T4)** antes de rodar.

## 1. Verificar GPU

In [ ]:
!nvidia-smi
import torch; print('CUDA:', torch.cuda.is_available())

## 2. Obter o código
Troque a URL pelo seu repositório no GitHub.

In [ ]:
REPO_URL = 'https://github.com/matosod1as/sin393-rice-disease-classification.git'
import os
if not os.path.exists('sin393-rice-disease-classification'):
    !git clone $REPO_URL
%cd sin393-rice-disease-classification
!pip -q install -r requirements.txt

## 3. Baixar o dataset do Kaggle
O `kagglehub` baixa datasets públicos. Se pedir autenticação, crie um token em *kaggle.com → Settings → API → Create New Token* e use `kagglehub.login()`.

In [ ]:
import kagglehub
DATA_DIR = kagglehub.dataset_download('anshulm257/rice-disease-dataset')
print(DATA_DIR)
!find "$DATA_DIR" -maxdepth 3 -type d | head -30

## 4. Rodar os 4 experimentos
Divisão estratificada 70/15/15 (`train_test_split`, semente 42), remoção de duplicatas antes da divisão, AdamW (lr=1e-4), até 20 épocas com *early stopping* (paciência 5) pelo F1-macro da validação. O teste só é usado na avaliação final.

Tempo estimado em uma T4: ~40–60 min no total.

In [ ]:
!python main.py --data-dir "$DATA_DIR" --epochs 20 --batch-size 32 --workers 2

## 5. Resultados no conjunto de teste

In [ ]:
import pandas as pd
s = pd.read_csv('results/summary.csv')
cols = ['experiment','accuracy','precision_macro','recall_macro','f1_macro','precision_weighted','recall_weighted','f1_weighted','n_errors','best_epoch','train_time_min','params_M']
s[cols].round(4)

In [ ]:
from IPython.display import Image, display
for f in ['class_distribution','dataset_samples','curves_all','cm_resnet50_aug','cm_resnet50_noaug','cm_vit_b_16_aug','cm_vit_b_16_noaug']:
    print(f); display(Image(f'paper/figures/{f}.png', width=600))

## 6. Análise qualitativa (acertos em verde, erros em vermelho; V = verdadeira, P = predita)

In [ ]:
import json
print(json.dumps(json.load(open('results/error_analysis.json')), indent=2, ensure_ascii=False))
for e in ['resnet50_aug','vit_b_16_aug']:
    print(e); display(Image(f'paper/figures/qualitative_{e}.png', width=750))

## 7. Baixar resultados para o artigo
O zip traz `results/`, `paper/generated/` (tabelas e números) e `paper/figures/`. Extraia na raiz do repositório (ou suba para o Overleaf) e faça commit.

In [ ]:
!zip -qr resultados_sin393.zip results paper/generated paper/figures -x '*.pt'
from google.colab import files
files.download('resultados_sin393.zip')